In [ ]:
# %% [markdown]
# # 🧠 Advanced Fear Detection with Vision Transformer (ViT‑Large)
# ## Multi‑Dataset Emotion Recognition (AffectNet + RAF‑DB + FER2013)
# 
# **Model:** ViT‑Large (`google/vit-large-patch16-224-in21k`) fine‑tuned on 7 emotion classes  
# **Focus:** Fear detection performance  
# **Outputs:** Best model, evaluation metrics, confusion matrix
# 
# ---

# %% [code]
# ─── 1. Install & Import Dependencies ──────────────────────────────────────────
!pip install -q transformers datasets accelerate scikit-learn opencv-python matplotlib seaborn

import os
import json
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, average_precision_score
)
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import shutil
import random
from sklearn.model_selection import train_test_split
from PIL import Image

# For ViT
from transformers import ViTForImageClassification, ViTImageProcessor

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

# %% [markdown]
# ## 2. Configuration

# %% [code]
# Paths to attached datasets (Kaggle mount points)
AFFECTNET_ROOT = "/kaggle/input/datasets/mstjebashazida/affectnet"
RAFDB_ROOT     = "/kaggle/input/datasets/shuvoalok/raf-db-dataset"
FER2013_ROOT   = "/kaggle/input/datasets/msambare/fer2013"

# Output directory for merged dataset
MERGED_ROOT = "/kaggle/working/emotion_combined"
os.makedirs(MERGED_ROOT, exist_ok=True)

# Standard 7 emotion classes
CLASSES = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)
FEAR_IDX = CLASS_TO_IDX['fear']

# Training hyperparameters (larger ViT requires smaller batch size)
BATCH_SIZE = 32          # Reduced for ViT‑Large (16GB GPU)
EPOCHS = 17             # Extended fine‑tuning
LEARNING_RATE = 1e-4     # Lower base LR, will use warm‑up
WEIGHT_DECAY = 1e-4
PATIENCE = 6             # Early stopping patience
WARMUP_EPOCHS = 3        # Linear warm‑up for first 3 epochs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# %% [markdown]
# ## 3. Dataset Merging (AffectNet + RAF‑DB + FER2013)
# 
# Collects all images, standardises classes, splits 70/15/15, and creates symlinks.

# %% [code]
def collect_affectnet():
    """Collect AffectNet images from Train/Test folders (skip contempt)."""
    data = []
    base = os.path.join(AFFECTNET_ROOT, 'archive (3)')
    for split in ['Train', 'Test']:
        split_path = os.path.join(base, split)
        if not os.path.exists(split_path):
            continue
        for class_name in os.listdir(split_path):
            if class_name == 'contempt':
                continue
            if class_name not in CLASSES:
                continue
            class_dir = os.path.join(split_path, class_name)
            if not os.path.isdir(class_dir):
                continue
            for img_file in os.listdir(class_dir):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(class_dir, img_file)
                    data.append((img_path, class_name))
    print(f"AffectNet: {len(data)} images")
    return data

def collect_rafdb():
    """RAF-DB: images are in DATASET/train/1/, DATASET/train/2/, etc. Map folder number to class."""
    data = []
    folder_to_class = {
        '1': 'surprise', '2': 'fear', '3': 'disgust', '4': 'happy',
        '5': 'sad', '6': 'angry', '7': 'neutral'
    }
    for split in ['train', 'test']:
        split_path = os.path.join(RAFDB_ROOT, 'DATASET', split)
        if not os.path.exists(split_path):
            continue
        for folder_num, class_name in folder_to_class.items():
            folder_path = os.path.join(split_path, folder_num)
            if not os.path.isdir(folder_path):
                continue
            for img_file in os.listdir(folder_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(folder_path, img_file)
                    data.append((img_path, class_name))
    print(f"RAF-DB: {len(data)} images")
    return data

def collect_fer2013():
    """FER2013 already has train/test splits with class subfolders."""
    data = []
    for split in ['train', 'test']:
        split_path = os.path.join(FER2013_ROOT, split)
        if not os.path.exists(split_path):
            continue
        for class_name in CLASSES:
            class_dir = os.path.join(split_path, class_name)
            if not os.path.isdir(class_dir):
                continue
            for img_file in os.listdir(class_dir):
                if img_file.endswith('.jpg'):
                    img_path = os.path.join(class_dir, img_file)
                    data.append((img_path, class_name))
    print(f"FER2013: {len(data)} images")
    return data

# Collect all
all_data = []
all_data.extend(collect_affectnet())
all_data.extend(collect_rafdb())
all_data.extend(collect_fer2013())
print(f"Total collected: {len(all_data)} images")

# Stratified split
from collections import defaultdict
class_to_items = defaultdict(list)
for img_path, cls in all_data:
    class_to_items[cls].append((img_path, cls))

train_items, val_items, test_items = [], [], []
for cls, items in class_to_items.items():
    # 70% train, 15% val, 15% test
    train, temp = train_test_split(items, test_size=0.3, random_state=42)
    val, test = train_test_split(temp, test_size=0.5, random_state=42)
    train_items.extend(train)
    val_items.extend(val)
    test_items.extend(test)
print(f"Train: {len(train_items)}, Val: {len(val_items)}, Test: {len(test_items)}")

# Create symlinks (saves disk space)
def link_items(items, split_name):
    split_dir = os.path.join(MERGED_ROOT, split_name)
    for img_path, cls in tqdm(items, desc=f"Linking {split_name}"):
        class_dir = os.path.join(split_dir, cls)
        os.makedirs(class_dir, exist_ok=True)
        fname = os.path.basename(img_path)
        dest = os.path.join(class_dir, fname)
        # Avoid duplicate filenames
        if os.path.exists(dest):
            base, ext = os.path.splitext(fname)
            counter = 1
            while os.path.exists(os.path.join(class_dir, f"{base}_{counter}{ext}")):
                counter += 1
            dest = os.path.join(class_dir, f"{base}_{counter}{ext}")
        try:
            os.symlink(img_path, dest)
        except:
            shutil.copy2(img_path, dest)

link_items(train_items, 'train')
link_items(val_items,   'val')
link_items(test_items,  'test')
print(f"✅ Merged dataset ready at {MERGED_ROOT}")

# Quick class count
print("\nClass distribution after merging:")
for cls in CLASSES:
    cnt = len(list(Path(MERGED_ROOT).rglob(f"*/{cls}/*.jpg")))
    print(f"  {cls}: {cnt}")

# %% [markdown]
# ## 4. Data Loaders with Augmentation

# %% [code]
# Image processor for ViT-Large
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

class EmotionDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        self.samples = []
        for cls in CLASSES:
            class_dir = Path(root_dir) / split / cls
            if not class_dir.exists():
                continue
            for img_path in class_dir.glob("*.jpg"):
                self.samples.append((str(img_path), CLASS_TO_IDX[cls]))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.fromarray(plt.imread(img_path)).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = EmotionDataset(MERGED_ROOT, 'train', transform=train_transforms)
val_dataset   = EmotionDataset(MERGED_ROOT, 'val',   transform=val_transforms)
test_dataset  = EmotionDataset(MERGED_ROOT, 'test',  transform=val_transforms)

# Weighted sampler for class imbalance
train_labels = [label for _, label in train_dataset.samples]
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]
sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

# %% [markdown]
# ## 5. Vision Transformer Model (ViT‑Large)

# %% [code]
class EmotionViT(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.vit = ViTForImageClassification.from_pretrained(
            'google/vit-base-patch16-224-in21k',
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        # Get the hidden size from the model config
        hidden_size = self.vit.config.hidden_size
        # Remove the original classifier
        self.vit.classifier = nn.Identity()
        # Replace with a custom head
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values):
        outputs = self.vit(pixel_values)
        # After removing the classifier, outputs.logits contains the features
        features = outputs.logits   # shape: (batch, hidden_size)
        return self.head(features)

    def fear_score(self, x):
        logits = self.forward(x)
        probs = F.softmax(logits, dim=-1)
        return probs[:, FEAR_IDX]
        


model = EmotionViT(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Cosine annealing scheduler with warm-up (manual warm-up in training loop)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS)

# %% [markdown]
# ## 6. Training Loop with Warm‑up and Early Stopping

# %% [code]
def train_epoch(loader):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    for images, labels in tqdm(loader, desc="Train"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc

@torch.no_grad()
def eval_epoch(loader):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(loader, desc="Val"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item()
        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    # Fear-specific metrics
    fear_true = [1 if l == FEAR_IDX else 0 for l in all_labels]
    fear_pred = [1 if p == FEAR_IDX else 0 for p in all_preds]
    fear_probs = [p[FEAR_IDX] for p in all_probs]
    fear_f1 = f1_score(fear_true, fear_pred)
    fear_auc = roc_auc_score(fear_true, fear_probs) if len(set(fear_true)) > 1 else 0.0
    return total_loss / len(loader), acc, f1, fear_f1, fear_auc

best_val_f1 = 0.0
patience_counter = 0
train_losses, val_losses = [], []

# Linear warm-up
warmup_steps = WARMUP_EPOCHS * len(train_loader)
global_step = 0

for epoch in range(1, EPOCHS+1):
    # Apply warm-up learning rate
    if epoch <= WARMUP_EPOCHS:
        lr_scale = epoch / WARMUP_EPOCHS
        for param_group in optimizer.param_groups:
            param_group['lr'] = LEARNING_RATE * lr_scale
    else:
        scheduler.step()   # Cosine annealing after warm-up

    train_loss, train_acc = train_epoch(train_loader)
    val_loss, val_acc, val_f1, fear_f1, fear_auc = eval_epoch(val_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:2d}/{EPOCHS} | LR {current_lr:.2e} | "
          f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
          f"Val Loss {val_loss:.4f} Acc {val_acc:.4f} F1 {val_f1:.4f} | "
          f"Fear F1 {fear_f1:.4f} AUC {fear_auc:.4f}")

    # Save best based on fear F1
    if fear_f1 > best_val_f1:
        best_val_f1 = fear_f1
        patience_counter = 0
        torch.save(model.state_dict(), '/kaggle/working/emotion_vit_best.pt')
        print(f"  ✅ New best model saved (Fear F1 = {fear_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

# %% [markdown]
# ## 7. Evaluation on Test Set

# %% [code]
# Load best model
model.load_state_dict(torch.load('/kaggle/working/emotion_vit_best.pt'))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(DEVICE)
        logits = model(images)
        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

# Overall metrics
test_acc = accuracy_score(all_labels, all_preds)
test_f1_macro = f1_score(all_labels, all_preds, average='macro')
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Macro F1 : {test_f1_macro:.4f}")

# Fear-specific metrics
fear_true = [1 if l == FEAR_IDX else 0 for l in all_labels]
fear_pred = [1 if p == FEAR_IDX else 0 for p in all_preds]
fear_probs = [p[FEAR_IDX] for p in all_probs]
fear_f1 = f1_score(fear_true, fear_pred)
fear_auc = roc_auc_score(fear_true, fear_probs)
fear_ap = average_precision_score(fear_true, fear_probs)
print(f"Fear class   : F1={fear_f1:.4f}, AUC={fear_auc:.4f}, AP={fear_ap:.4f}")

# Per-class report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.savefig('/kaggle/working/confusion_matrix.png', bbox_inches='tight')
plt.show()

# Save metrics to JSON
results = {
    'test_accuracy': test_acc,
    'test_f1_macro': test_f1_macro,
    'fear_f1': fear_f1,
    'fear_auc': fear_auc,
    'fear_ap': fear_ap,
    'classification_report': classification_report(all_labels, all_preds, target_names=CLASSES, output_dict=True)
}
with open('/kaggle/working/evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\n✅ Results saved to /kaggle/working/evaluation_results.json")

# %% [markdown]
# ## 8. Download Results
# 
# The best model and evaluation files are in `/kaggle/working/`. You can download them directly from the Kaggle UI or use the links below.

# %% [code]
from IPython.display import FileLink
print("Download best model:")
display(FileLink('/kaggle/working/emotion_vit_best.pt'))
print("\nDownload evaluation results:")
display(FileLink('/kaggle/working/evaluation_results.json'))
print("\nDownload confusion matrix:")
display(FileLink('/kaggle/working/confusion_matrix.png'))